# SRCNN — Super-Resolution Convolutional Neural Network

## Objašnjenje arhitekture, propagacije, loss funkcije i optimizacije

Ovaj notebook opisuje kako radi naš SRCNN model za poboljšanje rezolucije slika (super-resolution).  
Model radi u **YCbCr** prostoru boja — neuronska mreža obrađuje samo **Y kanal** (luminansu), dok se Cb i Cr (hromantski kanali) uveličavaju klasičnom bicubic interpolacijom.

## 1. Uvoz biblioteka

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Za prikaz slika u notebook-u
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

## 2. Arhitektura SRCNN modela

SRCNN se sastoji od **tri konvoluciona sloja** i **rezidualne (preskočne) veze**:

| Sloj | Kernel | Kanali | Aktivacija | Opis |
|------|--------|--------|------------|------|
| conv1 | 9×9 | 1 → 64 | ReLU | Ekstrakcija osobina (patch extraction) |
| conv2 | 5×5 | 64 → 32 | ReLU | Nelinearno preslikavanje (non-linear mapping) |
| conv3 | 5×5 | 32 → 1 | ❌ Nema | Rekonstrukcija (reconstruction) |

**Padding** je postavljen tako da izlazna slika ima iste dimenzije kao ulazna (same-padding).

### Rezidualna veza (Skip Connection)
Umesto da mreža direktno predviđa HR sliku, ona uči **rezidual** — razliku između interpolirane i originalne slike:

$$\hat{y} = \text{clamp}(x + f(x),\ 0,\ 1)$$

gde je $x$ ulazna slika (bicubic uveličana), a $f(x)$ izlaz konvolucionih slojeva.

In [ ]:
# Definicija SRCNN modela — identična kodu u src/neural_network/srcnn_model.py

class SRCNN(nn.Module):
    def __init__(self):
        super().__init__()

        # Tri konvoluciona sloja
        self.conv1 = nn.Conv2d(1, 64, kernel_size=9, padding=4)   # Ekstrakcija osobina
        self.conv2 = nn.Conv2d(64, 32, kernel_size=5, padding=2)  # Nelinearno preslikavanje
        self.conv3 = nn.Conv2d(32, 1, kernel_size=5, padding=2)   # Rekonstrukcija

        # Inicijalizacija težina
        nn.init.kaiming_normal_(self.conv1.weight, nonlinearity='relu')
        nn.init.kaiming_normal_(self.conv2.weight, nonlinearity='relu')
        nn.init.zeros_(self.conv3.weight)  # Zero-init: na početku f(x)=0, pa je output=input
        nn.init.zeros_(self.conv3.bias)

    def forward(self, x):
        residual = x                        # Sačuvaj ulaz za preskočnu vezu
        out = F.relu(self.conv1(x))          # Sloj 1 + ReLU
        out = F.relu(self.conv2(out))        # Sloj 2 + ReLU
        out = self.conv3(out)                # Sloj 3 (linearni, bez aktivacije)
        return torch.clamp(residual + out, 0.0, 1.0)  # Rezidualna veza + ograničenje na [0,1]

model = SRCNN()
print(model)
print(f"\nUkupno parametara: {sum(p.numel() for p in model.parameters()):,}")

## 3. Vizualizacija arhitekture

Sledeći dijagram prikazuje protok podataka kroz mrežu:

In [ ]:
# Vizualizacija arhitekture SRCNN-a

fig, ax = plt.subplots(1, 1, figsize=(14, 5))
ax.set_xlim(0, 14)
ax.set_ylim(0, 5)
ax.axis('off')
ax.set_title('SRCNN arhitektura sa rezidualnom vezom', fontsize=16, fontweight='bold')

# Boje
colors = {'input': '#4ECDC4', 'conv': '#FF6B6B', 'relu': '#FFE66D', 'output': '#95E1D3'}

# Blokovi
blocks = [
    (1, 2, 1.5, 1.5, 'Ulaz\n(Y kanal)\n1×H×W', colors['input']),
    (3.2, 2, 1.5, 1.5, 'Conv1\n9×9\n1→64', colors['conv']),
    (5.0, 2, 1.0, 1.5, 'ReLU', colors['relu']),
    (6.5, 2, 1.5, 1.5, 'Conv2\n5×5\n64→32', colors['conv']),
    (8.3, 2, 1.0, 1.5, 'ReLU', colors['relu']),
    (9.8, 2, 1.5, 1.5, 'Conv3\n5×5\n32→1', colors['conv']),
    (12.0, 2, 1.5, 1.5, 'Izlaz\nclamp\n[0,1]', colors['output']),
]

for x, y, w, h, text, color in blocks:
    rect = mpatches.FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.1",
                                     facecolor=color, edgecolor='black', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2, text, ha='center', va='center', fontsize=9, fontweight='bold')

# Strelice između blokova
arrow_pairs = [
    (2.5, 3.2), (4.7, 5.0), (6.0, 6.5), (8.0, 8.3), (9.3, 9.8), (11.3, 12.0)
]
for x1, x2 in arrow_pairs:
    ax.annotate('', xy=(x2, 2.75), xytext=(x1, 2.75),
                arrowprops=dict(arrowstyle='->', lw=2, color='black'))

# Rezidualna veza (skip connection) — luk iznad
ax.annotate('',
            xy=(12.0, 3.5), xytext=(2.5, 3.5),
            arrowprops=dict(arrowstyle='->', lw=2.5, color='#2C3E50',
                           connectionstyle='arc3,rad=-0.4', linestyle='--'))
ax.text(7.0, 4.6, 'Rezidualna veza (skip connection): output = input + f(input)',
        ha='center', va='center', fontsize=11, fontstyle='italic', color='#2C3E50')

# + oznaka
ax.text(11.65, 3.0, '+', ha='center', va='center', fontsize=18, fontweight='bold', color='#2C3E50')

plt.tight_layout()
plt.show()

## 4. Forward Propagation (Prolaz unapred)

Forward propagation je proces prosleđivanja ulaznih podataka kroz mrežu da bi se dobio izlaz.

### Korak po korak:

**1. Ulaz:** Bicubic-uveličana Y slika (luminansa), normalizovana na [0, 1], dimenzija `1×1×H×W`

**2. Conv1 + ReLU** — Ekstrakcija osobina:
- 64 filtera veličine 9×9 prelaze preko slike i detektuju lokalne teksture, ivice itd.
- **ReLU** ($f(x) = \max(0, x)$) uvodi nelinearnost — bez nje bi čitava mreža bila ekvivalentna jednoj linearnoj transformaciji.

**3. Conv2 + ReLU** — Nelinearno preslikavanje:
- Kombinuje 64 mapa osobina u 32 kompaktnijih. Ovo je "učenje" — mreža uči kako da iz detektovanih osobina rekonstruiše detalje.

**4. Conv3 (linearna)** — Rekonstrukcija:
- Iz 32 mape generiše jednokanalni izlaz — **rezidual** (koliko svaki piksel treba korigovati).
- **Nema aktivacione funkcije** jer rezidual može biti i pozitivan (posvetliti piksel) i negativan (potamniti piksel).

**5. Rezidualna veza + Clamp:**
$$\hat{y} = \text{clamp}(x + f(x),\ 0,\ 1)$$

Prednost: mreža treba da nauči samo **malu korekciju**, a ne kompletnu rekonstrukciju slike.

In [ ]:
# Demonstracija forward propagacije na random ulazu

model = SRCNN()

# Simulirani ulaz: 1 slika, 1 kanal (Y), 33x33 piksela
dummy_input = torch.rand(1, 1, 33, 33)
print(f"Ulaz:              shape={dummy_input.shape}, min={dummy_input.min():.3f}, max={dummy_input.max():.3f}")

# Praćenje međurezultata korak po korak
with torch.no_grad():
    residual = dummy_input

    # Sloj 1: Conv1 + ReLU
    after_conv1 = F.relu(model.conv1(dummy_input))
    print(f"Posle Conv1+ReLU:  shape={after_conv1.shape}, min={after_conv1.min():.3f}, max={after_conv1.max():.3f}")

    # Sloj 2: Conv2 + ReLU
    after_conv2 = F.relu(model.conv2(after_conv1))
    print(f"Posle Conv2+ReLU:  shape={after_conv2.shape}, min={after_conv2.min():.3f}, max={after_conv2.max():.3f}")

    # Sloj 3: Conv3 (bez aktivacije)
    after_conv3 = model.conv3(after_conv2)
    print(f"Posle Conv3:       shape={after_conv3.shape}, min={after_conv3.min():.3f}, max={after_conv3.max():.3f}")

    # Rezidualna veza + clamp
    output = torch.clamp(residual + after_conv3, 0.0, 1.0)
    print(f"Konačni izlaz:     shape={output.shape}, min={output.min():.3f}, max={output.max():.3f}")

    # Pošto je conv3 zero-inicijalizovan, rezidual je ~0
    print(f"\nRezidual (conv3 izlaz) — max aps. vrednost: {after_conv3.abs().max():.6f}")
    print("→ Zero-init znači da je na početku output ≈ input (identitet)")

## 5. Loss funkcija — MSE (Mean Squared Error)

**Mean Squared Error** (srednja kvadratna greška) meri prosečnu razliku između predviđenih i stvarnih piksela:

$$\mathcal{L}_{MSE} = \frac{1}{N} \sum_{i=1}^{N} (\hat{y}_i - y_i)^2$$

gde su:
- $\hat{y}_i$ — predviđena vrednost piksela (izlaz mreže)
- $y_i$ — stvarna vrednost piksela (iz originalne HR slike)
- $N$ — ukupan broj piksela u patch-u

### Zašto MSE?
- MSE **penalizuje veća odstupanja kvadratno** — jedno veliko odstupanje je gore od mnogo malih
- Direktno optimizuje **PSNR** (Peak Signal-to-Noise Ratio), jer je:

$$\text{PSNR} = 10 \cdot \log_{10}\!\left(\frac{MAX^2}{MSE}\right) = 20 \cdot \log_{10}\!\left(\frac{MAX}{\sqrt{MSE}}\right)$$

gde je $MAX$ maksimalna vrednost piksela (1.0 za normalizovane slike).

In [ ]:
# Demonstracija MSE loss-a i veze sa PSNR-om

criterion = nn.MSELoss()

# Simulirani primer: predviđeni vs. stvarni patch
torch.manual_seed(42)
y_pred = torch.rand(1, 1, 8, 8)          # Predviđeni pikseli
y_true = y_pred + 0.05 * torch.randn_like(y_pred)  # Stvarni (slični, ali ne isti)
y_true = torch.clamp(y_true, 0, 1)

# Računanje MSE
loss = criterion(y_pred, y_true)
print(f"MSE Loss: {loss.item():.6f}")

# Veza sa PSNR-om (za normalizovane slike, MAX=1.0)
psnr_value = 10 * torch.log10(1.0 / loss)
print(f"PSNR:     {psnr_value.item():.2f} dB")
print(f"\n→ Manji MSE = Veći PSNR = Bolja rekonstrukcija")

# Vizualizacija: kako veličina greške utiče na PSNR
mse_values = np.logspace(-5, -1, 100)
psnr_values = 10 * np.log10(1.0 / mse_values)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].semilogy(psnr_values, mse_values, 'b-', linewidth=2)
axes[0].set_xlabel('PSNR (dB)')
axes[0].set_ylabel('MSE (log skala)')
axes[0].set_title('Odnos MSE i PSNR')
axes[0].grid(True, alpha=0.3)
axes[0].axvline(x=30, color='r', linestyle='--', alpha=0.5, label='30 dB (dobar)')
axes[0].axvline(x=25, color='orange', linestyle='--', alpha=0.5, label='25 dB (OK)')
axes[0].legend()

# Vizualni primer razlike
axes[1].bar(['Mala greška\n(PSNR≈30dB)', 'Srednja greška\n(PSNR≈25dB)', 'Velika greška\n(PSNR≈20dB)'],
            [0.001, 0.00316, 0.01],
            color=['#2ecc71', '#f39c12', '#e74c3c'])
axes[1].set_ylabel('MSE')
axes[1].set_title('Kategorije kvaliteta rekonstrukcije')

plt.tight_layout()
plt.show()

## 6. Backward Propagation (Prolaz unazad)

Backward propagation (backpropagation) računamo automatski pozivom `loss.backward()`. PyTorch koristi **chain rule** (pravilo lanca) da izračuna gradijent loss funkcije po svakom parametru mreže.

### Kako radi:

**1.** Kreće se od loss-a i ide unazad kroz mrežu:

$$\frac{\partial \mathcal{L}}{\partial \theta} = \frac{\partial \mathcal{L}}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial h_3} \cdot \frac{\partial h_3}{\partial h_2} \cdot \frac{\partial h_2}{\partial h_1} \cdot \frac{\partial h_1}{\partial \theta}$$

gde su $h_1, h_2, h_3$ izlazi konvolucionih slojeva, a $\theta$ parametri (težine i biasi).

**2. Gradijent kroz ReLU:**
$$\frac{\partial \text{ReLU}(x)}{\partial x} = \begin{cases} 1 & \text{ako } x > 0 \\ 0 & \text{ako } x \leq 0 \end{cases}$$

ReLU propušta gradijent gde je aktivacija bila pozitivna, a blokira gde je bila nula.

**3. Rezidualna veza pomaže:**

Zahvaljujući skip connection-u, gradijent ima **direktan put** od izlaza do ulaza (zaobilazi konvolucione slojeve). Ovo smanjuje problem **nestajućih gradijenata** (vanishing gradients).

**4. Gradient Clipping:**

Koristimo `clip_grad_norm_(max_norm=1.0)` — ako ukupna norma gradijenata prekorači 1.0, svi se proporcionalno smanje. Sprečava **eksploziju gradijenata**.

In [ ]:
# Demonstracija backward propagacije

model = SRCNN()
criterion = nn.MSELoss()

# Ulaz i cilj
x = torch.rand(1, 1, 33, 33, requires_grad=False)
y_true = torch.rand(1, 1, 33, 33)

# Forward pass
y_pred = model(x)
loss = criterion(y_pred, y_true)
print(f"Loss pre backprop: {loss.item():.6f}")

# Pre backward-a: gradijenti su None
print(f"\nGradijenti pre backward-a:")
for name, param in model.named_parameters():
    print(f"  {name:20s} | grad = {param.grad}")

# Backward pass — izračunava gradijente
loss.backward()

# Posle backward-a: gradijenti su izračunati
print(f"\nGradijenti posle backward-a:")
for name, param in model.named_parameters():
    grad_norm = param.grad.norm().item() if param.grad is not None else 0
    print(f"  {name:20s} | shape={str(list(param.shape)):20s} | grad norma = {grad_norm:.6f}")

# Gradient clipping
total_norm_before = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
print(f"\nUkupna norma gradijenata: {total_norm_before:.4f}")
print(f"(Ako > 1.0, gradijenti se proporcionalno smanjuju)")

## 7. Optimizacija — Adam optimizer

**Adam** (Adaptive Moment Estimation) je optimizacioni algoritam koji za svaki parametar $\theta$ održava dva pokretna proseka:

### Momentum (prvi moment — $m_t$):
$$m_t = \beta_1 \cdot m_{t-1} + (1 - \beta_1) \cdot g_t$$

Eksponencijalni prosek gradijenata. Smanjuje oscilacije i ubrzava konvergenciju u pravcu konzistentnog gradijenta.

### RMSProp (drugi moment — $v_t$):
$$v_t = \beta_2 \cdot v_{t-1} + (1 - \beta_2) \cdot g_t^2$$

Eksponencijalni prosek **kvadrata** gradijenata. Normalizuje korak — parametri sa velikim gradijentima dobijaju manji korak i obrnuto.

### Ažuriranje parametra:
$$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1 - \beta_2^t}$$

$$\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{\hat{v}_t} + \epsilon} \cdot \hat{m}_t$$

### Hiperparametri koje koristimo:
| Parametar | Vrednost | Opis |
|-----------|----------|------|
| $\eta$ (lr) | $10^{-4}$ za conv1/conv2, $10^{-5}$ za conv3 | Learning rate (diferencijalni) |
| $\beta_1$ | 0.9 | Momentum faktor |
| $\beta_2$ | 0.999 | RMSProp faktor |
| $\epsilon$ | $10^{-8}$ | Numerička stabilnost |

### Diferencijalni learning rate:
Conv3 (rekonstrukcioni sloj) uči **10× sporije** od ostalih slojeva, jer je izlazni sloj osetljiviji — malene promene u njemu direktno menjaju izlaz slike.

### Learning Rate Scheduler:
**ReduceLROnPlateau** — ako se validation loss ne poboljša 3 uzastopne epohe, learning rate se prepolovi ($\times 0.5$). Ovo omogućava grubo podešavanje na početku i fino podešavanje kasnije.

In [ ]:
# Demonstracija optimizacije — jedan korak treninga

model = SRCNN()

# Diferencijalni learning rate (kao u train.py)
optimizer = optim.Adam([
    {'params': model.conv1.parameters(), 'lr': 1e-4},       # Conv1: lr = 0.0001
    {'params': model.conv2.parameters(), 'lr': 1e-4},       # Conv2: lr = 0.0001
    {'params': model.conv3.parameters(), 'lr': 1e-4 * 0.1}, # Conv3: lr = 0.00001 (10x sporije)
])

# Learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

criterion = nn.MSELoss()

# Simulacija jednog koraka treninga
x = torch.rand(4, 1, 33, 33)       # Mini-batch: 4 patcha
y_true = torch.rand(4, 1, 33, 33)

# Sačuvaj težine pre ažuriranja
w1_before = model.conv1.weight.data[0, 0, 0, :3].clone()

# 1. Forward pass
y_pred = model(x)

# 2. Izračunaj loss
loss = criterion(y_pred, y_true)

# 3. Backward pass (izračunaj gradijente)
optimizer.zero_grad()  # Resetuj gradijente na 0
loss.backward()

# 4. Gradient clipping
nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

# 5. Ažuriraj parametre
optimizer.step()

# 6. Ažuriraj learning rate (na osnovu val loss-a)
scheduler.step(loss)

# Uporedi težine pre i posle
w1_after = model.conv1.weight.data[0, 0, 0, :3].clone()

print("=== Jedan korak optimizacije ===")
print(f"Loss: {loss.item():.6f}")
print(f"\nConv1 težine (prva 3 elementa prvog filtera):")
print(f"  Pre:   {w1_before.numpy()}")
print(f"  Posle: {w1_after.numpy()}")
print(f"  Δ:     {(w1_after - w1_before).numpy()}")
print(f"\nLearning rates:")
for i, pg in enumerate(optimizer.param_groups):
    layer = ['conv1', 'conv2', 'conv3'][i]
    print(f"  {layer}: {pg['lr']:.6f}")

## 8. Kompletni trening ciklus — sve zajedno

Svaka **epoha** treninga ponavlja sledeće korake za svaki mini-batch:

```
Za svaki mini-batch (lr_patch, hr_patch):
    1. Forward:    ŷ = model(lr_patch)           # Prosledi kroz mrežu
    2. Loss:       L = MSE(ŷ, hr_patch)          # Izračunaj grešku
    3. Zero grad:  optimizer.zero_grad()          # Resetuj prethodne gradijente
    4. Backward:   L.backward()                  # Izračunaj nove gradijente
    5. Clip:       clip_grad_norm_(max_norm=1.0)  # Ograniči gradijente
    6. Step:       optimizer.step()               # Ažuriraj parametre
```

Posle svake epohe:
- Evaluira se model na **validation setu** (bez ažuriranja parametara)
- **Scheduler** smanjuje lr ako nema poboljšanja
- **Early stopping** prekida trening ako nema napretka 10 epoha
- Čuva se **best model** (sa najnižim val loss-om)

In [ ]:
# Simulacija mini-treninga na sintetičkim podacima
# (bez stvarnih slika, samo za demonstraciju konvergencije)

model = SRCNN()
optimizer = optim.Adam([
    {'params': model.conv1.parameters(), 'lr': 1e-4},
    {'params': model.conv2.parameters(), 'lr': 1e-4},
    {'params': model.conv3.parameters(), 'lr': 1e-4 * 0.1},
])
criterion = nn.MSELoss()

# Kreiranje sintetičkog dataseta: ulaz + šum → cilj je ulaz (denoising zadatak)
torch.manual_seed(0)
clean = torch.rand(16, 1, 33, 33) * 0.8 + 0.1   # "Čist" signal
noisy = clean + 0.05 * torch.randn_like(clean)     # Sa šumom
noisy = torch.clamp(noisy, 0, 1)

# Trening petlja
losses = []
psnrs = []
num_epochs = 50

for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()

    output = model(noisy)
    loss = criterion(output, clean)
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    with torch.no_grad():
        mse_val = loss.item()
        psnr_val = 10 * np.log10(1.0 / max(mse_val, 1e-12))
        losses.append(mse_val)
        psnrs.append(psnr_val)

# Vizualizacija konvergencije
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(range(1, num_epochs+1), losses, 'b-', linewidth=2)
ax1.set_xlabel('Epoha')
ax1.set_ylabel('MSE Loss')
ax1.set_title('Smanjenje loss-a tokom treninga')
ax1.grid(True, alpha=0.3)
ax1.set_yscale('log')

ax2.plot(range(1, num_epochs+1), psnrs, 'g-', linewidth=2)
ax2.set_xlabel('Epoha')
ax2.set_ylabel('PSNR (dB)')
ax2.set_title('Povećanje PSNR-a tokom treninga')
ax2.grid(True, alpha=0.3)

plt.suptitle('Konvergencija modela na sintetičkim podacima', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Početni loss: {losses[0]:.6f} → Krajnji loss: {losses[-1]:.6f}")
print(f"Početni PSNR: {psnrs[0]:.2f} dB → Krajnji PSNR: {psnrs[-1]:.2f} dB")

## 9. Inicijalizacija težina

Pravilna inicijalizacija je ključna za stabilan trening:

| Sloj | Inicijalizacija | Razlog |
|------|----------------|--------|
| conv1 | **Kaiming Normal** | Optimalna za ReLU — održava varijansu aktivacija kroz slojeve |
| conv2 | **Kaiming Normal** | Isto kao conv1 |
| conv3 | **Zeros** (sve nule) | Na početku $f(x) = 0$, pa je izlaz = ulaz (identitet) |

### Kaiming inicijalizacija:
$$W \sim \mathcal{N}\left(0, \sqrt{\frac{2}{n_{in}}}\right)$$

gde je $n_{in}$ broj ulaznih kanala×kernel_width×kernel_height. Ovo obezbeđuje da varijansa aktivacija ostane konstantna kroz sve slojeve, sprečavajući nestajanje ili eksploziju signala.

### Zero inicijalizacija conv3:
Zahvaljujući rezidualnoj vezi, zero-init znači:
- Na početku: $\hat{y} = x + 0 = x$ (mreža je identitet)
- Trening polako uči korekcije od nule
- Sprečava da netrenirani model **pogorša** ulaznu sliku

In [ ]:
# Vizualizacija distribucije težina po slojevima

model = SRCNN()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for i, (name, param) in enumerate([(n, p) for n, p in model.named_parameters() if 'weight' in n]):
    weights = param.data.flatten().numpy()
    axes[i].hist(weights, bins=50, color=['#3498db', '#2ecc71', '#e74c3c'][i],
                 alpha=0.7, edgecolor='black', linewidth=0.5)
    axes[i].set_title(f'{name}\nμ={weights.mean():.4f}, σ={weights.std():.4f}', fontsize=11)
    axes[i].set_xlabel('Vrednost težine')
    axes[i].set_ylabel('Frekvencija')
    axes[i].axvline(x=0, color='black', linestyle='--', alpha=0.5)

plt.suptitle('Distribucija težina posle inicijalizacije', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("conv1 i conv2: Kaiming Normal — široka distribucija oko nule")
print("conv3: Zeros — sve težine su tačno 0 (identitet na početku)")

## 10. Rezime — ceo pipeline

```
┌─────────────────────────────────────────────────────────────────┐
│                    TRENING PIPELINE                             │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  1. PRIPREMA PODATAKA                                           │
│     ├── Učitaj LR (degradiranu) i HR (originalnu) sliku        │
│     ├── Konvertuj u YCbCr → uzmi samo Y kanal                  │
│     ├── Uveličaj LR bicubic interpolacijom (×3)                │
│     └── Iseci random patch 33×33                               │
│                                                                 │
│  2. FORWARD PROPAGATION                                         │
│     ├── Conv1 (9×9, 1→64) + ReLU                               │
│     ├── Conv2 (5×5, 64→32) + ReLU                              │
│     ├── Conv3 (5×5, 32→1) — linearni                           │
│     └── Output = clamp(input + conv_output, 0, 1)              │
│                                                                 │
│  3. LOSS                                                        │
│     └── MSE = mean((predicted - target)²)                      │
│                                                                 │
│  4. BACKWARD PROPAGATION                                        │
│     └── Izračunaj ∂L/∂θ za sve parametre (chain rule)          │
│                                                                 │
│  5. OPTIMIZACIJA                                                │
│     ├── Gradient clipping (max_norm=1.0)                       │
│     ├── Adam update (sa momentum i adaptivnim lr)              │
│     └── LR scheduler: prepolovi lr ako val loss stagnira       │
│                                                                 │
│  6. REGULARIZACIJA                                              │
│     ├── Early stopping (patience=10 epoha)                     │
│     ├── Data augmentation (flip, rotacija)                     │
│     └── Diferencijalni lr (conv3 uči 10× sporije)             │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

### Pokretanje treninga:
```bash
python -m src.neural_network.train --epochs 50 --scale 3 --patch 33 --batch-size 64
```

### Pokretanje benchmark-a:
```bash
python scripts/benchmark_methods.py --scale 3
```